# Preprocessing snRNA-seq Batches and Combining Multi-Batch AnnData

QC-filters each batch of CellBender-denoised counts using the same strategy,
concatenates all batches, identifies and saves the final merged object.

## Setup

In [ ]:
import os

os.environ["NUMBA_CACHE_DIR"] = "/path/to/tmp"
os.environ["MPLCONFIGDIR"] = "/path/to/tmp"

import anndata as ad
import matplotlib.pyplot as plt
import scanpy as sc

%matplotlib inline
sc.settings.verbosity = 1  # errors (0), warnings (1), info (2), hints (3)
sc.settings.figdir = "./figures-cellbender/preprocessing/"
sc.settings.set_figure_params(dpi=80)

## Config

In [ ]:
# input batches: add as many as you have, in any order
batch_input_paths = [
    "/path/to/data/S1_counts_adata_lung01.h5ad",
    "/path/to/data/S1_counts_adata_lung02.h5ad",
    "/path/to/data/S1_counts_adata_lung03.h5ad",
]
batch_qc_output_dir = "/path/to/output/"

merged_output_path = "/path/to/output/S2_counts_preprocessed_adata_merge.h5ad"

# QC thresholds (applied identically to every batch)
min_genes = 500
min_counts = 1000
max_percent_mito = 0.05  # keep <10% cutoff was considered too permissive for early apoptotic cells; 5% used instead

os.makedirs(batch_qc_output_dir, exist_ok=True)

## 1. QC-filter each batch

Same filtering strategy (min genes, min UMI counts, max mito%) applied to every batch.

In [ ]:
def qc_filter_batch(adata, min_genes, min_counts, max_percent_mito, plot=True):
    """Apply the standard per-batch QC pipeline: min-genes filter, min-UMI-count
    filter, and max-mitochondrial-fraction filter, with optional diagnostic plots."""
    if plot:
        sc.pl.scatter(adata, x="n_counts", y="percent_mito", show=True)
        sc.pl.scatter(adata, x="n_counts", y="n_genes", show=True)
        plt.hist(adata.obs["n_genes"], bins=100, range=[0, 10000])
        plt.xlim(0, 10000)
        plt.axvline(min_genes, linestyle="--", color="red")
        plt.axvline(200, linestyle="--", color="green")
        plt.show()

    sc.pp.filter_cells(adata, min_genes=min_genes)
    print("Number of cells after min_genes filter: {:d}".format(adata.n_obs))

    if plot:
        plt.hist(adata.obs["n_counts"], bins=200)
        plt.xlim(0, 20000)
        plt.axvline(min_counts, linestyle="--", color="red")
        plt.show()

    sc.pp.filter_cells(adata, min_counts=min_counts)
    print("Number of cells after min_counts filter: {:d}".format(adata.n_obs))

    if plot:
        plt.hist(adata.obs["percent_mito"], bins=100, cumulative=True)
        plt.axvline(0.2, linestyle="--", color="red")
        plt.axvline(0.1, linestyle="--", color="darkred")
        plt.axvline(0.05, linestyle="--", color="blue")
        plt.axhline(adata.n_obs * 0.99, linestyle="-", color="green")
        plt.show()

    # keep cells below max_percent_mito. A stringent 5% cutoff is used (rather than 10%)
    # to filter out likely-dead/apoptotic cells while still tolerating some
    # early-apoptotic signal that may carry real biological information.
    adata = adata[adata.obs["percent_mito"] < max_percent_mito, :].copy()

    print("Total number of cells: {:d}".format(adata.n_obs))
    print("Total number of genes: {:d}".format(adata.n_vars))

    if plot:
        sc.pl.violin(adata, ["n_genes", "percent_mito"], groupby="sample", rotation=90)
        sc.pl.violin(adata, ["n_counts"], groupby="sample", rotation=90)
        sc.pl.violin(adata, ["n_genes", "n_counts", "percent_mito"], groupby="group2", rotation=90)

    return adata

In [ ]:
batch_adatas = []
for batch_path in batch_input_paths:
    print(f"Processing {batch_path}")
    adata = sc.read(batch_path)
    adata = qc_filter_batch(adata, min_genes=min_genes, min_counts=min_counts, max_percent_mito=max_percent_mito)

    batch_name = os.path.splitext(os.path.basename(batch_path))[0]
    out_path = os.path.join(batch_qc_output_dir, f"{batch_name}_S2_preprocessed.h5ad")
    adata.write(out_path)

    batch_adatas.append(adata)

## 2. Concatenate QC-filtered batches

In [ ]:
adata_merge = ad.concat(batch_adatas, join="outer", index_unique=None)
adata_merge

In [ ]:
sc.set_figure_params(figsize=(8, 8))
sc.pl.violin(adata_merge, ["n_genes"], groupby="sample", rotation=90)
sc.pl.violin(adata_merge, ["percent_mito"], groupby="sample", rotation=90)
sc.pl.violin(adata_merge, ["n_counts"], groupby="sample", rotation=90)
sc.pl.violin(adata_merge, ["n_genes", "n_counts", "percent_mito"], groupby="group2", rotation=90)
sc.pl.violin(adata_merge, ["n_genes"], groupby="donor", rotation=90)

### Save merged batches

In [ ]:
adata_merge.write(merged_output_path)

## 3. Marker-gene QC: cluster and check for liver-biopsy samples

Normalizes, clusters, and visualizes hepatocyte/cholangiocyte/LSEC/Kupffer/NK markers to identify samples from liver-biopsy. 

In [ ]:
def cluster_and_plot_markers(adata, groupby, marker_genes, n_neighbors=40, resolution=0.3, save_suffix=None):
    """Normalize, cluster (neighbors + UMAP + Leiden), and plot marker gene
    expression on UMAP, colored by `groupby`."""
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.neighbors(adata, n_neighbors=n_neighbors)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=resolution)

    save_arg = f"_{save_suffix}.pdf" if save_suffix else None
    sc.pl.umap(
        adata, color=marker_genes + [groupby],
        use_raw=False, color_map="OrRd", size=10, save=save_arg,
    )
    return adata

In [ ]:
# marker gene sets used for contamination QC
liver_qc_markers = [
    "AFP", "TTR", "CYP3A4", "ASGR1", "ASGR2",  # hepatocytes
    "KRT19", "SOX9", "CFTR",                    # cholangiocytes
    "LYVE1", "STAB1", "STAB2",                  # LSECs
    "CLEC4F", "MARCO", "CD68",                  # Kupffer cells
    "NCAM1", "ZEB2", "KLRD1", "KLRB1",           # NK cells
]

broad_marker_genes = {
    "Tumour": {"NAPSA", "NKX2-1", "CEACAM5", "CECR2", "MUC21"},
    "Liver": {"AFP", "TTR", "CYP3A4", "ASGR1", "ASGR2"},
}


In [ ]:
adata_merge = cluster_and_plot_markers(
    adata_merge, groupby="ID", marker_genes=liver_qc_markers, save_suffix="all_samples"
)

In [ ]:
sc.pl.umap(
    adata_merge,
    color=["group2", "batchseq", "n_genes", "scrublet_score", "percent_mito", "ID"],
    color_map="OrRd", ncols=4, wspace=0.3,
)

In [ ]:
sc.pl.dotplot(adata_merge, broad_marker_genes, groupby="sample", use_raw=False,
               standard_scale="var", cmap="Blues", show=False, swap_axes=True)
sc.pl.dotplot(adata_merge, broad_marker_genes, groupby="ID", use_raw=False,
               standard_scale="var", cmap="Blues", show=False, swap_axes=True)

### Save filtered merge

In [ ]:
adata_merge.write(merged_regress_output_path)